# **03 Historical OISST Climatology and Event Data**
This notebook downloads daily NOAA OISST observations for the California Current study region (25–48°N, 135–115°W), constructs a historical daily mean and 90th-percentile SST climatology, and downloads OISST for the two study events.

Historical baseline implemented: 1982–2011 (30 years)

Outputs: data/processed/oisst_climatology_1982_2011.nc and annual regional OISST NetCDF files under data/processed/oisst_yearly/

Requirements: xarray, numpy, pandas, netCDF4.

Paths assume the notebook runs from the repository's notebooks/ directory

## 1. Setup and dependencies

In [ ]:
from pathlib import Path
import time

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

baseline_years = range(1982, 2012)  # 1982–2011 inclusive
# later on, oisst for 2012 is also downloaded
# so all oisst baseline years available are from 1982 to 2012 inclusive
event_years = [2013, 2014, 2015, 2016, 2019, 2020]
oisst_folder = Path("../data/processed/oisst_yearly")
oisst_folder.mkdir(parents=True, exist_ok=True)

## 2. Download annual OISST data for the historical baseline
Daily OISST files are accessed remotely and cropped to the study region. Longitude is represented on the 0–360° grid, so 135–115°W corresponds to 225–245°E. The download function retries failed monthly requests and skips annual files that have the expected number of days.

Approximate time required to download: 2 hours

In [5]:
# function to download OISST data for a given year
def download_oisst_year(year):

    output_file = oisst_folder / f"oisst_cc_{year}.nc"

    # skip years already downloaded
    if output_file.exists():
        try:
            with xr.open_dataset(output_file) as existing:
                expected_days = (
                    pd.Timestamp(f"{year + 1}-01-01")
                    - pd.Timestamp(f"{year}-01-01")
                ).days

                if existing.sizes["time"] == expected_days:
                    print(f"{year}: already saved, skipping")
                    return
        except Exception:
            print(f"{year}: existing file invalid, retrying")

    url = (
        "https://psl.noaa.gov/thredds/dodsC/"
        f"Datasets/noaa.oisst.v2.highres/sst.day.mean.{year}.nc"
    )

    print(f"\nDownloading {year}...")

    months = []

    with xr.open_dataset(url, engine="netcdf4") as ds:

        regional = ds["sst"].sel(
            lat=slice(25, 48),
            lon=slice(225, 245)
        )

        for month in range(1, 13):

            start = f"{year}-{month:02d}-01"
            end = (
                pd.Timestamp(start)
                + pd.offsets.MonthEnd(0)
            ).strftime("%Y-%m-%d")

            for attempt in range(3):
                try:
                    monthly = regional.sel(
                        time=slice(start, end)
                    ).load()

                    months.append(monthly)
                    print(f"  Month {month:02d} completed")
                    break

                except Exception as e:
                    print(
                        f"  Month {month:02d} failed "
                        f"(attempt {attempt + 1}): {e}"
                    )

                    if attempt == 2:
                        raise

                    time.sleep(5)

    yearly = xr.concat(months, dim="time")

    # check the number of days before saving
    expected_days = (
        pd.Timestamp(f"{year + 1}-01-01")
        - pd.Timestamp(f"{year}-01-01")
    ).days

    assert yearly.sizes["time"] == expected_days

    # save via temporary file to avoid incomplete checkpoints
    temp_file = output_file.with_suffix(".tmp.nc")

    yearly.to_netcdf(temp_file)
    temp_file.replace(output_file)

    print(f"{year}: saved successfully!")

In [ ]:
for year in baseline_years:
    download_oisst_year(year)

print("\nAll baseline years downloaded!")

1982: already saved, skipping
1983: already saved, skipping
1984: already saved, skipping
1985: already saved, skipping
1986: already saved, skipping
1987: already saved, skipping
1988: already saved, skipping
1989: already saved, skipping

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
1990: saved successfully!

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
1991: saved successfully!

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 c

### 2a. Download 2012 File
Initially, we downloaded from 1982 to 2011 but later realised that we need to inspect up to the year 2012. Therefore, additional download for the year 2012 is conducted.

In [ ]:
# Download 2012 OISST
download_oisst_year(2012)


  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
2012: saved successfully!


In [7]:
print((oisst_folder / "oisst_cc_2012.nc").exists())

True


### 2b. Repair and validate 2008 file
The June 2008 mean showed zero, which should not happen. This is a check to repair the 2008 file so all 30 baseline files are valid and non-zero.



In [ ]:
# Repair the 2008 OISST file without losing the original

import shutil
import numpy as np
import xarray as xr

original = oisst_folder / "oisst_cc_2008.nc"
backup = oisst_folder / "oisst_cc_2008_backup.nc"

if not original.exists():
    raise FileNotFoundError(f"Cannot find {original}")

if backup.exists():
    raise FileExistsError(
        f"Backup already exists: {backup}. "
        "Check it before running this cell again."
    )

# 1. Preserve original data
shutil.copy2(original, backup)
print("Original 2008 file backed up.")

# 2. Remove only the active 2008 file so the function
#    does not skip the download
original.unlink()

try:
    download_oisst_year(2008)

    # 3. Validate the new file
    with xr.open_dataset(original) as ds:
        sst = ds["sst"]

        if ds.sizes["time"] != 366:
            raise ValueError("2008 file does not contain 366 days.")

        monthly_means = (
            sst.groupby("time.month")
            .mean(skipna=True)
            .compute()
        )

        june = sst.sel(
            time=slice("2008-06-01", "2008-06-30")
        )

        june_mean = float(june.mean())
        june_max = float(june.max())

        print("\n2008 monthly SST means:")
        print(monthly_means.values.round(3))

        print("\nJune 2008 mean:", june_mean)
        print("June 2008 maximum:", june_max)

        if not np.isfinite(june_mean) or june_mean == 0:
            raise ValueError(
                "June 2008 still has invalid SST values."
            )

    print("\n2008 downloaded and passed basic checks!")
    print("Original backup retained at:", backup)

except Exception:
    # Restore original if download or validation fails
    shutil.copy2(backup, original)
    print("Download/validation failed; original restored.")
    raise


Original 2008 file backed up.

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
2008: saved successfully!

2008 monthly SST means:
[[[20.142 20.041 19.992 ... 17.246 17.382 17.544]
  [20.183 20.062 19.974 ... 17.288 17.477 17.633]
  [20.147 20.008 19.881 ... 17.357 17.573 17.711]
  ...
  [ 7.167  7.222  7.234 ...    nan    nan    nan]
  [ 7.12   7.102  7.058 ...    nan    nan    nan]
  [ 7.029  6.974  6.9   ...    nan    nan    nan]]

 [[19.548 19.574 19.53  ... 17.134 17.174 17.142]
  [19.425 19.449 19.422 ... 16.925 16.998 17.018]
  [19.264 19.256 19.257 ... 16.828 16.913 16.96 ]
  ...
  [ 6.668  6.635  6.62  ...    nan    nan    nan]
  [ 6.6    6.551  6.513 ...    nan    nan    nan]
  [ 6.496  6.463  6.432 ...    nan    nan    nan]]

 [[19.619 19.612 19.506 ... 17.167 17.227 17.234]


In [ ]:
# verifying data for 2008
with xr.open_dataset(oisst_folder / "oisst_cc_2008.nc") as ds:
    monthly_means = (
        ds["sst"]
        .groupby("time.month")
        .mean(dim=("time", "lat", "lon"), skipna=True)
    )

    for month, temp in zip(monthly_means.month.values, monthly_means.values):
        print(f"Month {month:02d}: {temp:.3f}°C")

Month 01: 13.912°C
Month 02: 13.295°C
Month 03: 13.287°C
Month 04: 13.366°C
Month 05: 14.084°C
Month 06: 15.445°C
Month 07: 17.365°C
Month 08: 18.485°C
Month 09: 18.177°C
Month 10: 17.664°C
Month 11: 16.724°C
Month 12: 15.357°C


## 3. Load only the historical baseline years
The climatology must be calculated using baseline years only, even if the same folder also contains OISST files downloaded for the marine heatwave events.

In [ ]:
# building the historical SST climatology
baseline_files = [oisst_folder / f"oisst_cc_{year}.nc" for year in baseline_years]
missing = [str(p) for p in baseline_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing historical OISST files: {missing}")

datasets = []

for file in baseline_files:
    with xr.open_dataset(file) as ds:
        datasets.append(ds["sst"].load())

sst_all = xr.concat(datasets, dim="time").sortby("time")

## 4. Standardise the 365-day Calendar
Remove February 29 and assign each remaining observation to a common day-of-year index. This enables historical values from different years to be pooled by calendar day.

In [ ]:
# remove February 29
not_leap_day = ~(
    (sst_all.time.dt.month == 2) &
    (sst_all.time.dt.day == 29)
)

sst_no_leap = sst_all.sel(time=not_leap_day)

# assign a consistent 1–365 calendar-day index
month_day = sst_no_leap.time.dt.strftime("%m-%d").values

calendar = pd.date_range(
    "2001-01-01", "2001-12-31", freq="D"
)

day_lookup = {
    date.strftime("%m-%d"): i + 1
    for i, date in enumerate(calendar)
}

day_index = np.array([
    day_lookup[date] for date in month_day
])

sst_no_leap = sst_no_leap.assign_coords(
    climatology_day=("time", day_index)
)

## 5. Calculate the daily historical mean and 90th percentile
For each calendar day, observations within an 11-day circular window (±5 days, including across year boundaries) are pooled across the historical baseline. The mean and 90th percentile are calculated separately at each OISST grid cell.

In [ ]:
# calculate the historical mean and 90th percentile
clim_means = []
clim_p90 = []

days = sst_no_leap["climatology_day"].values

for day in range(1, 366):

    # circular distance handles year boundaries
    distance = np.abs(days - day)
    circular_distance = np.minimum(distance, 365 - distance)

    # 11-day window: day - 5 through day + 5
    mask = circular_distance <= 5

    window_sst = sst_no_leap.isel(time=mask)

    mean = window_sst.mean(dim="time", skipna=True)
    p90 = window_sst.quantile(
        0.90, dim="time", skipna=True
    ).squeeze(drop=True)

    clim_means.append(mean)
    clim_p90.append(p90)

    if day % 50 == 0:
        print(f"Processed {day}/365 calendar days")

mean_sst = xr.concat(clim_means, dim="dayofyear")
p90_sst = xr.concat(clim_p90, dim="dayofyear")

mean_sst = mean_sst.assign_coords(
    dayofyear=np.arange(1, 366)
)

p90_sst = p90_sst.assign_coords(
    dayofyear=np.arange(1, 366)
)

print(mean_sst.shape)
print(p90_sst.shape)

Processed 50/365 calendar days
Processed 100/365 calendar days
Processed 150/365 calendar days
Processed 200/365 calendar days
Processed 250/365 calendar days
Processed 300/365 calendar days
Processed 350/365 calendar days
(365, 92, 80)
(365, 92, 80)


## 6. Smooth the climatological thresholds
Apply a circular 31-day moving mean to both daily climatologies to reduce day-to-day variability while preserving continuity across the end of the calendar year.

In [22]:
def smooth_circular(data, window=31):

    half = window // 2

    extended = xr.concat(
        [
            data.isel(dayofyear=slice(-half, None)),
            data,
            data.isel(dayofyear=slice(0, half))
        ],
        dim="dayofyear",
        coords="minimal",
        compat="override"
    )

    smoothed = (
        extended
        .rolling(dayofyear=window, center=True)
        .mean()
        .isel(dayofyear=slice(half, half + 365))
    )

    return smoothed.assign_coords(
        dayofyear=np.arange(1, 366)
    )


mean_sst_smooth = smooth_circular(mean_sst)
p90_sst_smooth = smooth_circular(p90_sst)

print(mean_sst_smooth.shape)
print(p90_sst_smooth.shape)

(365, 92, 80)
(365, 92, 80)


## 7. Save the historical climatology
Export the daily historical mean and 90th-percentile threshold for use by subsequent marine heatwave detection notebooks.

In [11]:
import shutil
from pathlib import Path

output_path = Path("../data/processed/oisst_climatology_1982_2011.nc")
backup_path = output_path.with_name(
    "oisst_climatology_1982_2011_backup.nc"
)

if output_path.exists() and not backup_path.exists():
    shutil.copy2(output_path, backup_path)
    print("Climatology backup created.")

Climatology backup created.


In [ ]:
# saving historical mean and 90th percentile threshold into a single file
climatology = xr.Dataset(
    {
        "mean_sst": mean_sst_smooth.astype("float32"),
        "p90_sst": p90_sst_smooth.astype("float32")
    }
)

climatology.attrs["baseline_period"] = "1982–2011"
climatology.attrs["description"] = (
    "California Current OISST daily climatology, "
    "11-day window, 31-day smoothing"
)
climatology.attrs["units"] = "degrees Celsius"

output_path = Path(
    "../data/processed/oisst_climatology_1982_2011.nc"
)

climatology.to_netcdf(output_path)

print("Saved:", output_path)

Saved: ..\data\processed\oisst_climatology_1982_2011.nc


## 8. Validate saved climatology

In [12]:
with xr.open_dataset(
    "../data/processed/oisst_climatology_1982_2011.nc"
) as ds:
    print(ds)
    print("Baseline:", ds.attrs["baseline_period"])
    print("Days:", ds.sizes["dayofyear"])
    print("Mean SST contains valid values:",
          bool(ds["mean_sst"].notnull().any()))
    print("P90 contains valid values:",
          bool(ds["p90_sst"].notnull().any()))

<xarray.Dataset> Size: 21MB
Dimensions:    (dayofyear: 365, lat: 92, lon: 80)
Coordinates:
  * dayofyear  (dayofyear) int32 1kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365
  * lat        (lat) float32 368B 25.12 25.38 25.62 25.88 ... 47.38 47.62 47.88
  * lon        (lon) float32 320B 225.1 225.4 225.6 225.9 ... 244.4 244.6 244.9
    quantile   float64 8B ...
Data variables:
    mean_sst   (dayofyear, lat, lon) float32 11MB ...
    p90_sst    (dayofyear, lat, lon) float32 11MB ...
Attributes:
    baseline_period:  1982–2011
    description:      California Current OISST daily climatology, 11-day wind...
    units:            degrees Celsius
Baseline: 1982–2011
Days: 365
Mean SST contains valid values: True
P90 contains valid values: True


## 9. Download OISST Data for the Marine Heatwave Events
Download daily regional OISST observations for the 2013–2016 Blob and the 2019–2020 marine heatwave. These files are used in subsequent notebooks to compare observed SST conditions against the historical climatology.

In [ ]:
for year in event_years:
    download_oisst_year(year)

print("All event-period OISST data downloaded!")


  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
2013: saved successfully!

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
2014: saved successfully!

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 completed
  Month 09 completed
  Month 10 completed
  Month 11 completed
  Month 12 completed
2015: saved successfully!

  Month 01 completed
  Month 02 completed
  Month 03 completed
  Month 04 completed
  Month 05 completed
  Month 06 completed
  Month 07 completed
  Month 08 comp

## 10. Summary and downstream use
This notebook produces a daily historical SST mean and P90 threshold from the 1982–2011 OISST baseline, using an 11-day climatological window followed by 31-day smoothing. The saved NetCDF file supports subsequent RQ1 detection and model comparison. Event-year OISST files are downloaded separately for the Blob (2013–2016) and the later marine heatwave (2019–2020).